In [1]:
import pandas as pd
from pathlib import Path

file_path = Path("../data/processed/model_eligible.csv")
model_eligible = pd.read_csv(file_path)

print(model_eligible.shape)
print(model_eligible["CHURNED"].value_counts())
print(model_eligible.isna().sum())

(2225, 30)
CHURNED
0    1563
1     662
Name: count, dtype: int64
CUSTOMER_ID                    0
SIGNUP_DATE                    0
AGE                            0
GENDER                         0
CITY                           0
MEMBERSHIP_TIER                0
MARKETING_OPT_IN               0
PREFERRED_STORE_ID             0
PREFERRED_CATEGORY             0
HOME_STORE_PRICE_TIER          0
TRANSACTIONS_PRE               0
TOTAL_SPEND_PRE                0
APP_SESSIONS_PRE               0
EMAILS_OPENED_PRE              0
COUPONS_REDEEMED_PRE           0
SUPPORT_TICKETS_PRE            0
TRANSACTIONS_AVG_PRE           0
TOTAL_SPEND_AVG_PRE            0
AVG_BASKET_VALUE_AVG_PRE       0
PROMO_TXN_PCT_AVG_PRE          0
APP_SESSIONS_AVG_PRE           0
EMAILS_OPENED_AVG_PRE          0
SUPPORT_TICKETS_AVG_PRE        0
ACTIVE_MONTHS_PRE              0
RECENT_TRANSACTIONS          449
RECENT_SPEND                 449
RECENT_APP_SESSIONS          449
RECENT_SUPPORT_TICKETS       449
CHURNED    

In [3]:
missing = model_eligible.isna().sum()
print(missing[missing > 0])

RECENT_TRANSACTIONS       449
RECENT_SPEND              449
RECENT_APP_SESSIONS       449
RECENT_SUPPORT_TICKETS    449
dtype: int64


In [4]:
print(model_eligible.dtypes.to_string())

CUSTOMER_ID                    int64
SIGNUP_DATE                      str
AGE                          float64
GENDER                           str
CITY                             str
MEMBERSHIP_TIER                  str
MARKETING_OPT_IN               int64
PREFERRED_STORE_ID             int64
PREFERRED_CATEGORY               str
HOME_STORE_PRICE_TIER            str
TRANSACTIONS_PRE             float64
TOTAL_SPEND_PRE              float64
APP_SESSIONS_PRE             float64
EMAILS_OPENED_PRE            float64
COUPONS_REDEEMED_PRE         float64
SUPPORT_TICKETS_PRE          float64
TRANSACTIONS_AVG_PRE         float64
TOTAL_SPEND_AVG_PRE          float64
AVG_BASKET_VALUE_AVG_PRE     float64
PROMO_TXN_PCT_AVG_PRE        float64
APP_SESSIONS_AVG_PRE         float64
EMAILS_OPENED_AVG_PRE        float64
SUPPORT_TICKETS_AVG_PRE      float64
ACTIVE_MONTHS_PRE            float64
RECENT_TRANSACTIONS          float64
RECENT_SPEND                 float64
RECENT_APP_SESSIONS          float64
R

In [5]:
# Make a copy so the original dataset stays unchanged
df = model_eligible.copy()

# Convert signup date into a datetime
df["SIGNUP_DATE"] = pd.to_datetime(df["SIGNUP_DATE"])

# Create customer tenure in days
df["TENURE_DAYS"] = (
    pd.Timestamp("2024-04-01") - df["SIGNUP_DATE"]
).dt.days

# Remove columns we should not use as model inputs
X = df.drop(columns=[
    "CUSTOMER_ID",
    "SIGNUP_DATE",
    "CHURNED",
    "INSUFFICIENT_HISTORY_FLAG"
])

# Target: what we want to predict
y = df["CHURNED"]

print("Features:", X.shape)
print("Target:", y.shape)
print("Target values:")
print(y.value_counts())

Features: (2225, 27)
Target: (2225,)
Target values:
CHURNED
0    1563
1     662
Name: count, dtype: int64


In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

print("\nTraining churn rate:")
print(y_train.mean())

print("\nTesting churn rate:")
print(y_test.mean())

Training data: (1780, 27)
Testing data: (445, 27)

Training churn rate:
0.29775280898876405

Testing churn rate:
0.2966292134831461


In [7]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", len(categorical_cols))

Numeric columns: 22
Categorical columns: 5


In [8]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.pipeline import Pipeline

# Baseline model
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

# Logistic Regression model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# Compare results
for name, pred in [
    ("Baseline", baseline_pred),
    ("Logistic Regression", y_pred)
]:
    print(f"\n{name}")
    print("Accuracy :", round(accuracy_score(y_test, pred), 3))
    print("Precision:", round(precision_score(y_test, pred, zero_division=0), 3))
    print("Recall   :", round(recall_score(y_test, pred, zero_division=0), 3))
    print("F1-score :", round(f1_score(y_test, pred, zero_division=0), 3))


Baseline
Accuracy : 0.703
Precision: 0.0
Recall   : 0.0
F1-score : 0.0

Logistic Regression
Accuracy : 0.836
Precision: 0.761
Recall   : 0.652
F1-score : 0.702


In [9]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Confusion Matrix:
[[286  27]
 [ 46  86]]

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.91      0.89       313
           1       0.76      0.65      0.70       132

    accuracy                           0.84       445
   macro avg       0.81      0.78      0.79       445
weighted avg       0.83      0.84      0.83       445



In [10]:
from sklearn.metrics import confusion_matrix, classification_report

# Get predicted churn probabilities
y_prob = model.predict_proba(X_test)[:, 1]

# Predict churn if probability is at least 0.4
y_pred_04 = (y_prob >= 0.4).astype(int)

print(confusion_matrix(y_test, y_pred_04))
print(classification_report(y_test, y_pred_04))

[[272  41]
 [ 37  95]]
              precision    recall  f1-score   support

           0       0.88      0.87      0.87       313
           1       0.70      0.72      0.71       132

    accuracy                           0.82       445
   macro avg       0.79      0.79      0.79       445
weighted avg       0.83      0.82      0.83       445



In [11]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

results = []

for threshold, pred in [(0.5, y_pred), (0.4, y_pred_04)]:
    results.append({
        "Threshold": threshold,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1-score": f1_score(y_test, pred)
    })

results_df = pd.DataFrame(results)
print(results_df.round(3))

   Threshold  Accuracy  Precision  Recall  F1-score
0        0.5     0.836      0.761   0.652     0.702
1        0.4     0.825      0.699   0.720     0.709


In [12]:
recent_cols = [
    "RECENT_TRANSACTIONS",
    "RECENT_SPEND",
    "RECENT_APP_SESSIONS",
    "RECENT_SUPPORT_TICKETS"
]

missing_recent = X[recent_cols].isna().all(axis=1)

print(pd.crosstab(
    missing_recent,
    y,
    rownames=["All recent features missing"],
    colnames=["CHURNED"]
))

CHURNED                         0    1
All recent features missing           
False                        1556  220
True                            7  442


In [13]:
# Remove the four recent-activity features
recent_cols = [
    "RECENT_TRANSACTIONS",
    "RECENT_SPEND",
    "RECENT_APP_SESSIONS",
    "RECENT_SUPPORT_TICKETS"
]

X_without_recent = X.drop(columns=recent_cols)

# Split using the same settings
X_train_nr, X_test_nr, y_train_nr, y_test_nr = train_test_split(
    X_without_recent, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Build a fresh preprocessing pipeline for these columns
num_cols_nr = X_train_nr.select_dtypes(include="number").columns
cat_cols_nr = X_train_nr.select_dtypes(exclude="number").columns

preprocessor_nr = ColumnTransformer([
    ("numeric", numeric_pipeline, num_cols_nr),
    ("categorical", categorical_pipeline, cat_cols_nr)
])

model_nr = Pipeline([
    ("preprocessor", preprocessor_nr),
    ("classifier", LogisticRegression(max_iter=1000))
])

model_nr.fit(X_train_nr, y_train_nr)
pred_nr = model_nr.predict(X_test_nr)

print(classification_report(y_test_nr, pred_nr))

              precision    recall  f1-score   support

           0       0.86      0.93      0.89       313
           1       0.79      0.64      0.70       132

    accuracy                           0.84       445
   macro avg       0.82      0.78      0.80       445
weighted avg       0.84      0.84      0.84       445



In [14]:
print("With recent features:")
print(confusion_matrix(y_test, y_pred))

print("\nWithout recent features:")
print(confusion_matrix(y_test_nr, pred_nr))

With recent features:
[[286  27]
 [ 46  86]]

Without recent features:
[[290  23]
 [ 48  84]]


In [15]:
from pathlib import Path

results_df.to_csv(
    Path("../data/processed/day3_model_comparison.csv"),
    index=False
)